
# 01 — The corrected benchmark

Every model on one rolling-origin fold schedule, with naive baselines, fold-level
spreads, and Diebold–Mariano tests under FDR control.

Three things this fixes relative to the earlier study: models were compared across
*different* train/test splits; no naive baseline was ever computed; and on ~300 test
points the RMSE gaps between models sit inside fold-to-fold noise.

**Read this notebook together with 04.** At one step ahead the learned models beat a
one-line baseline only modestly, and that is the honest result here. The case for
learning is made at the lead times an allocator actually needs, which is notebook 04.

In [1]:
# --- Bootstrap: works locally and on Colab ---------------------------------
# Locally this just finds the repository root. On Colab the repo is not on the VM
# yet, so it is cloned first. The repository is PRIVATE, which means the clone
# needs a GitHub token -- put one in Colab Secrets (the key icon in the left
# sidebar) under the name GH_TOKEN and enable notebook access. See docs/COLAB.md.
import os, sys, warnings
from pathlib import Path

warnings.filterwarnings("ignore")
REPO = "github.com/sad-code-at/bwalloc.git"

ROOT = Path.cwd()
while not (ROOT / "src" / "bwalloc").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent

if not (ROOT / "src" / "bwalloc").exists():
    target = Path("/content/bwalloc")
    if not (target / "src" / "bwalloc").exists():
        try:
            from google.colab import userdata
            token = userdata.get("GH_TOKEN")
        except Exception:
            token = None
        if not token:
            raise SystemExit(
                "Could not find the repository, and no GH_TOKEN is available. "
                "On Colab: add a GitHub token in Secrets (the key icon) as "
                "GH_TOKEN, enable notebook access for this notebook, and re-run "
                "-- see docs/COLAB.md. Locally: run this notebook from inside "
                "the repository."
            )
        # The token never reaches stdout: git is quiet and errors are sanitised.
        rc = os.system(f"git clone -q https://{token}@{REPO} {target} 2>/dev/null")
        if rc != 0 or not (target / "src" / "bwalloc").exists():
            raise SystemExit(
                "git clone failed. Check that GH_TOKEN is valid, not expired, and "
                "has read access to this repository (Contents: Read)."
            )
    os.chdir(target)
    ROOT = target

sys.path.insert(0, str(ROOT / "src"))

try:
    import xgboost  # noqa: F401
except ImportError:
    !pip install -q xgboost

import numpy as np, pandas as pd, matplotlib.pyplot as plt
import bwalloc as bw
from bwalloc.plots import use_paper_style

bw.set_seed()
use_paper_style()
pd.set_option("display.width", 200)
RESULTS = ROOT / "experiments" / "results"
# Scratch output for the exploratory notebooks. Only 07_paper_figures writes into
# paper/figures -- otherwise running notebook 00 or 05 silently overwrites a figure
# the paper cites, which is exactly the kind of drift this project exists to remove.
FIGURES = ROOT / "notebooks" / "figures"
FIGURES.mkdir(parents=True, exist_ok=True)
print("bwalloc", bw.__version__, "| results:", RESULTS)

bwalloc 0.1.0 | results: D:\L4-T-1\EEE 402\project\bwalloc\experiments\results


In [2]:

from bwalloc.baselines import SeasonalNaive, standard_baselines
from bwalloc.data import load, sampling_profile
from bwalloc.evaluate import beats_baseline, dm_matrix, run_backtest, summarise
from bwalloc.features import FeatureConfig, build_features
from bwalloc.models import default_point_models
from bwalloc.splits import describe_folds, rolling_origin

OPERATOR = "gp"          # switch to "robi" and re-run
N_FOLDS = 8

df = load(OPERATOR)
profile = sampling_profile(df)
X, y = build_features(df, profile, FeatureConfig())
folds = rolling_origin(len(y), n_folds=N_FOLDS)
describe_folds(folds, X.index)

,fold,n_train,n_calib,n_test,train_start,test_start,test_end
0,0,266,89,66,2025-04-05 10:28:00,2025-04-26 23:00:00,2025-04-30 19:45:00
1,1,316,105,66,2025-04-05 10:28:00,2025-04-30 21:10:00,2025-05-04 17:55:00
2,2,365,122,66,2025-04-05 10:28:00,2025-05-04 19:21:00,2025-05-08 16:06:00
3,3,415,138,66,2025-04-05 10:28:00,2025-05-08 17:31:00,2025-05-12 14:16:00
4,4,464,155,66,2025-04-05 10:28:00,2025-05-12 15:42:00,2025-05-16 12:27:00
5,5,514,171,66,2025-04-05 10:28:00,2025-05-16 13:52:00,2025-05-20 10:37:00
6,6,563,188,66,2025-04-05 10:28:00,2025-05-20 12:03:00,2025-05-24 08:48:00
7,7,613,204,71,2025-04-05 10:28:00,2025-05-24 10:13:00,2025-05-28 14:06:00


## Backtest

Every model and every baseline on identical folds.

In [3]:

baselines = standard_baselines(y.to_numpy(), profile.daily_period)
baselines.append(SeasonalNaive(y.to_numpy(), period=24))   # the lag the original used

per_fold, predictions = run_backtest(
    X, y, folds, models=default_point_models(),
    baselines=baselines, season_lag=profile.daily_period,
)
summary = beats_baseline(summarise(per_fold))
summary[["model", "rmse_mean", "rmse_std", "mae_mean", "mase_mean",
         "vs_persistence", "beats_persistence"]]

,model,rmse_mean,rmse_std,mae_mean,mase_mean,vs_persistence,beats_persistence
0,random_forest,10.410761,1.187650,7.172142,0.513663,0.141346,True
1,xgboost,10.797524,1.524965,7.672821,0.549659,0.109446,True
2,ridge,11.258809,0.877070,7.916516,0.567079,0.071401,True
3,persistence,12.124506,0.919605,6.473070,0.463599,0.000000,False
4,rolling_mean_17,16.162736,1.654161,12.504571,0.895473,-0.333063,False
5,train_mean,18.234407,1.955814,14.760692,1.057260,-0.503930,False
6,seasonal_naive_17,18.306637,1.650798,14.524147,1.040434,-0.509887,False
7,drift,23.967919,7.492340,19.670494,1.409514,-0.976816,False
8,seasonal_naive_24,29.740611,3.876509,25.268771,1.809248,-1.452934,False


**The gate.** A model that does not beat persistence on the same folds is not a result. `run_benchmark.py` exits non-zero if the top-ranked model fails this.

In [4]:

best = summary.iloc[0]
assert bool(best["beats_persistence"]), f"{best['model']} does not beat persistence"
print(f"best: {best['model']} — RMSE {best['rmse_mean']:.3f} ± {best['rmse_std']:.3f} "
      f"({best['vs_persistence']:+.1%} vs persistence)")

best: random_forest — RMSE 10.411 ± 1.188 (+14.1% vs persistence)


## Is the ranking real?

Diebold–Mariano across folds, Benjamini–Hochberg corrected because ten models is 45 comparisons. Most adjacent pairs are not separable on this much data — which is itself the finding.

In [5]:

dm = dm_matrix(predictions, horizon=1)
top = summary["model"].head(4).tolist()
dm[dm["model_a"].isin(top) & dm["model_b"].isin(top)][
    ["model_a", "model_b", "dm_stat", "p_value", "significant_fdr", "winner"]
]

,model_a,model_b,dm_stat,p_value,significant_fdr,winner
8,persistence,random_forest,3.874256,0.000120,True,random_forest
9,persistence,ridge,3.458450,0.000587,True,ridge
14,persistence,xgboost,2.682684,0.007531,True,xgboost
15,random_forest,ridge,-2.245216,0.025164,True,random_forest
20,random_forest,xgboost,-2.238665,0.025590,True,random_forest
25,ridge,xgboost,0.972274,0.331356,False,tie



## Feature ablation — a negative result, reported

`experiments/run_benchmark.py` runs the full ablation. Its conclusion is worth
stating plainly because it cuts against the correction:

**Correcting the feature design does not improve accuracy on either operator.** Tree
ensembles route around a mis-specified `lag_24` by leaning on `lag_1..3`, so the
sampling-rate error costs almost nothing in RMSE once a flexible model is used.

That does not make the correction pointless — it makes its value *interpretive*
rather than predictive. Every seasonal claim in the earlier study (STL, ACF, FFT,
"peak at f ≈ 0.0417 ⇒ 24-hour cycle") was stated on the wrong time axis, and the
seasonal-naive comparison in notebook 00 shows what that costs a model that cannot
route around it.

In [6]:

ablation = pd.read_csv(RESULTS / f"ablation_{OPERATOR}.csv")
ablation

,operator,ablation,n_features,best_model,rmse_mean,rmse_std
0,gp,lags_only,8,random_forest,9.980842,1.116067
1,gp,no_fourier,17,random_forest,9.997645,1.118071
2,gp,original_style,19,random_forest,10.109055,1.141777
3,gp,original_correct_period,19,random_forest,10.119683,1.167707
4,gp,no_context,16,random_forest,10.409982,1.242855
5,gp,full,25,random_forest,10.410761,1.187650
